# 17-Handling Imbalanced Data

In our previous lessons, we perfected the mathematical engine of our Neural Network. We optimized the gradients, scheduled the learning rates, and regularized the weights. We assumed that if the engine is perfect, the network will learn perfectly.

But there is a fatal flaw in that assumption: **The Data Itself.**

In the real world, data is rarely a perfectly balanced 50/50 split. If you are building an AI to detect fraudulent credit card transactions, 99.9% of transactions are perfectly legal, and 0.1% are fraud. If you feed this imbalanced data into a perfectly optimized Deep Learning model, the model will suffer from mathematical laziness.

Imbalanced data creates a phenomenon known as the **Accuracy Paradox**. If a network simply predicts "Legal" for every single transaction, it will be 99.9% accurate. The Loss Function (BCE) will be incredibly low, the Optimizer will be happy, and the network will completely fail to identify a single instance of fraud.

To fix this, we must force the network to care about the minority class. We do this through Data-Level Sampling and Algorithm-Level Loss Modification.

Let's set up our PyTorch environment to rebalance the scales of mathematics.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ PyTorch Class Imbalance Environment Ready.")

✅ PyTorch Class Imbalance Environment Ready.


# 1. Data-Level: The `WeightedRandomSampler`

The most intuitive way to fix imbalance is to fix the data before it enters the network.

If you have 9,900 images of Dogs and 100 images of Cats, you could artificially duplicate the Cat images (Oversampling) or delete 9,800 Dog images (Undersampling). However, manually duplicating data in RAM is highly inefficient for massive Deep Learning datasets.

Instead, PyTorch uses a **`WeightedRandomSampler`**. We leave the raw dataset exactly as it is. But when the `DataLoader` goes to construct a mini-batch of 32 images, we give it a weighted probability map. We tell it: *"You are 99 times more likely to pick a Cat image than a Dog image."*
The resulting mini-batch sent to the GPU will be a perfectly balanced 50/50 split of Cats and Dogs, even though the underlying dataset is 99/1.

# 2. Algorithm-Level: Class Weights

Sometimes, altering the data sampler is not enough, or it introduces too much repetitive noise (showing the network the exact same Cat image 99 times).

Instead of changing the data, we change the **Loss Function**. We keep the batches imbalanced, but we mathematically punish the network much harder if it gets the minority class wrong.

Recall the Categorical Cross-Entropy Loss:


$$L_{CCE} = - \sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

We introduce a class weight vector ($w_c$). If class 0 (Dog) has 9,900 samples, and class 1 (Cat) has 100 samples, we assign a massive weight multiplier to the Cat class.


$$L_{Weighted} = - \sum_{c=1}^{C} w_c \cdot y_c \log(\hat{y}_c)$$

If the network misclassifies a Dog, the loss goes up by $1$. If the network misclassifies a Cat, the loss goes up by $99$. The optimizer realizes that missing a Cat is a mathematical catastrophe, and violently shifts the weights to ensure it never happens again.

# 3. The State-of-the-Art: Focal Loss

In 2017, AI researchers at Facebook were trying to build an Object Detection model (RetinaNet). An image contains maybe 3 objects (hard minority class), and 100,000 pixels of empty background (easy majority class). Standard Class Weights failed because the sheer volume of "easy" background pixels still overwhelmed the loss gradient.

They invented **Focal Loss**. Instead of weighting based on *class frequency*, Focal Loss dynamically weights the loss based on the network's *confidence*.

$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$


*(Where $p_t$ is the probability the network assigns to the correct class).*

**The Genius of $\gamma$ (Gamma):**

* Imagine the network looks at a piece of blank sky (Background). It is 99% confident ($p_t = 0.99$) it is Background.
* In standard Cross-Entropy, $- \log(0.99)$ still generates a tiny bit of loss. Multiplied by 100,000 pixels, it overwhelms the network.
* With Focal Loss, we add the modulating factor $(1 - p_t)^\gamma$. If $\gamma = 2$, then $(1 - 0.99)^2 = 0.0001$.
* **The Result:** The loss for "easy, highly-confident" predictions is mathematically crushed to absolutely zero. The network is forced to focus 100% of its gradient updates on the "hard, low-confidence" minority examples.

# 4. Implementing Balanced Deep Learning in PyTorch

Let's simulate an extremely imbalanced dataset (95% Class 0, 5% Class 1). We will implement the PyTorch enterprise standard for resolving this: combining `WeightedRandomSampler` for the DataLoader, and `weight` tensors for the Loss Function.

In [2]:
# 1. Simulate a Heavily Imbalanced Dataset
num_majority = 9500 # Class 0
num_minority = 500  # Class 1

# Features
X_maj = torch.randn(num_majority, 10)
X_min = torch.randn(num_minority, 10) + 2.0 # Slightly shifted feature space
X = torch.cat([X_maj, X_min], dim=0)

# Labels
y_maj = torch.zeros(num_majority, dtype=torch.long)
y_min = torch.ones(num_minority, dtype=torch.long)
y = torch.cat([y_maj, y_min], dim=0)

dataset = TensorDataset(X, y)

print("--- 🚨 Dataset Imbalance Audit ---")
print(f"Class 0 (Majority): {num_majority} samples")
print(f"Class 1 (Minority): {num_minority} samples\n")

# 2. STRATEGY A: The WeightedRandomSampler
# First, calculate the class counts
class_counts = torch.tensor([num_majority, num_minority], dtype=torch.float32)

# Calculate the weight for each class (Inverse Frequency)
# Class 0 Weight: 1/9500 | Class 1 Weight: 1/500
class_weights = 1.0 / class_counts

# Assign the specific weight to every single sample in the dataset
sample_weights = class_weights[y]

# Create the Sampler
sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=len(sample_weights), 
    replacement=True # Must be True so it can pick the minority class multiple times!
)

# Create the DataLoader WITH the sampler
# Note: You cannot use shuffle=True when using a custom sampler!
balanced_loader = DataLoader(dataset, batch_size=32, sampler=sampler)

print("--- ⚖️ DataLoader Sampling Audit ---")
# Let's pull exactly ONE batch and check the distribution
X_batch, y_batch = next(iter(balanced_loader))
print(f"Batch Size: {len(y_batch)}")
print(f"Class 0 in Batch: {(y_batch == 0).sum().item()}")
print(f"Class 1 in Batch: {(y_batch == 1).sum().item()}")
print("Insight: Even though the dataset is 95% to 5%, the DataLoader mathematically forces the GPU batches to be roughly 50/50!\n")

# 3. STRATEGY B: Loss Function Class Weights
# If we didn't want to use the Sampler, we can pass weights directly to the Loss Function.
# We give Class 0 a weight of 1.0, and Class 1 a weight of 19.0 (since 9500/500 = 19)
loss_weights = torch.tensor([1.0, 19.0], dtype=torch.float32)

# Pass the weights into CrossEntropyLoss
weighted_criterion = nn.CrossEntropyLoss(weight=loss_weights)

print("--- ⚖️ Loss Function Weights Audit ---")
print(f"Loss multiplier for Class 0 mistakes: {loss_weights[0].item()}")
print(f"Loss multiplier for Class 1 mistakes: {loss_weights[1].item()}")
print("Insight: The optimizer will now prioritize fixing Minority Class errors 19 times more aggressively!")

--- 🚨 Dataset Imbalance Audit ---
Class 0 (Majority): 9500 samples
Class 1 (Minority): 500 samples

--- ⚖️ DataLoader Sampling Audit ---
Batch Size: 32
Class 0 in Batch: 16
Class 1 in Batch: 16
Insight: Even though the dataset is 95% to 5%, the DataLoader mathematically forces the GPU batches to be roughly 50/50!

--- ⚖️ Loss Function Weights Audit ---
Loss multiplier for Class 0 mistakes: 1.0
Loss multiplier for Class 1 mistakes: 19.0
Insight: The optimizer will now prioritize fixing Minority Class errors 19 times more aggressively!


## Real-World Use Case or Analogy:

Think of Imbalanced Data like **A Security Guard at a Diamond Mine**:

* **The Accuracy Paradox**: The mine employs 10,000 honest miners (Majority) and 1 thief (Minority). The Security Guard realizes that if he just waves everybody through the metal detector without checking them, he will be mathematically 99.99% accurate at his job. The mine owners praise his "accuracy", but diamonds keep getting stolen.
* **The Data Sampler (The Random Search Rule)**: Management intervenes. They institute a rule: 1 out of every 2 people the guard inspects *must* be someone who triggered a slight metal detector anomaly. Even though anomalous people are rare, the guard's actual "batch" of work is now an even 50/50 split of normal vs. suspicious people.
* **Class Weights (The Consequence Rule)**: Management tells the guard: "If you accidentally delay an honest miner, the penalty is a $\$1$ fine. If you accidentally let a thief through, the penalty is you go to prison for 20 years." The mathematical consequence of a False Negative (missing the minority class) is now so catastrophically high that the guard completely changes his behavior, hyper-focusing on anyone even remotely suspicious.